# Data Loading

In [19]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [12]:
# Load the article metadata
current_folder = Path.cwd()
project_root = current_folder.parent.parent.parent
DATA_DIR = project_root / "data" / "raw"
ARTICLES_PATH = os.path.join(DATA_DIR, "articles.csv")
CUSTOMERS_PATH = os.path.join(DATA_DIR, "customers.csv")
TRANSACTIONS_PATH = os.path.join(DATA_DIR, "transactions_train.csv")

# Ensure IDs are strings for safe joins
articles_df = pd.read_csv(ARTICLES_PATH, dtype={"article_id": "string"})
customers = pd.read_csv(CUSTOMERS_PATH, dtype={"customer_id": "string"})
transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    dtype={"customer_id": "string", "article_id": "string"},
    parse_dates=["t_dat"],
)

print("articles:", articles_df.shape)
print("customers:", customers.shape)
print("transactions:", transactions.shape)


articles: (105542, 25)
customers: (1371980, 7)
transactions: (31788324, 5)


In [ ]:
# Create Check Function
def check_image_exists(article_id):
    # Get the first 3 digit for the subfolder
    subfolder = article_id[:3]

    # Construct the image path
    image_path = os.path.join(DATA_DIR, "images", subfolder, f"{article_id}.jpg")

    return os.path.exists(image_path)

# Scan and Filter
print(f"Original articles count: {len(articles_df)}")
print("Scanning for images...")

# Check if images exist and filter the articles
articles_df["has_image"] = articles_df["article_id"].apply(check_image_exists)

# Filter articles that have images
clean_articles_df = articles_df[articles_df["has_image"]].copy()

# Drop the helper column
clean_articles_df.drop(columns=["has_image"], inplace=True)

print(f"Articles with images: {len(clean_articles_df)}")
print(f"Dropped articles without images: {len(articles_df) - len(clean_articles_df)}")

# Top-Bottom Filtering

In [22]:
# Ensure IDs are propperly formatted strings
articles_df["article_id"] = articles_df["article_id"].apply(lambda x: f"{int(x):010d}")

# Determine Gender
is_ladies_index = articles_df['index_group_name'].isin(['Ladieswear', 'Divided'])
is_ladies_dept = articles_df['department_name'].str.contains('Ladies|Women', case=False, na=False)
is_mens_index = articles_df['index_group_name'] == 'Menswear'
is_mens_dept = articles_df['department_name'].str.contains('Mens|Men', case=False, na=False)

print("Articles with Ladieswear index:", is_ladies_index.sum())
print("Articles with Ladieswear department:", is_ladies_dept.sum())
print("Articles with Menswear index:", is_mens_index.sum())
print("Articles with Menswear department:", is_mens_dept.sum())

# Combine the rules
gender_conditions = [
    (is_ladies_index | is_ladies_dept),
    (is_mens_index | is_mens_dept)
]
gender_choices = ['Ladieswear', 'Menswear']

# Apply the logic
articles_df['broad_gender']= np.select(gender_conditions, gender_choices, default='other')

# Determine Type ('top' vs 'bottom')
type_conditions = [
    articles_df['product_group_name'] == 'Garment Upper body',
    articles_df['product_group_name'] == 'Garment Lower body'
]
type_choices = ['top', 'bottom']
articles_df['broad_type'] = np.select(type_conditions, type_choices, default='other')

# 4. Combine them
articles_df['combined_category'] = articles_df['broad_gender'] + ' ' + articles_df['broad_type']

# 5. Filter and Format
valid_items = articles_df[
    (articles_df['broad_gender'] != 'other') & 
    (articles_df['broad_type'] != 'other')
].copy()

category_edges = valid_items[['article_id', 'combined_category']].copy()
category_edges.columns = ['source', 'target']
category_edges['relation'] = 'has_category'

print(f"Generated {len(category_edges)} 'has_category' edges.")
print("\nCategory Distribution:")
print(category_edges['target'].value_counts())

Articles with Ladieswear index: 54886
Articles with Ladieswear department: 1901
Articles with Menswear index: 12553
Articles with Menswear department: 1755
Generated 41464 'has_category' edges.

Category Distribution:
target
Ladieswear top       21910
Ladieswear bottom     9484
Menswear top          7369
Menswear bottom       2701
Name: count, dtype: int64


# Best Matches Based on Transaction

In [25]:
# Create a Category Lookup Dictionary
# Maps '0123456789' -> 'womens top', 'mens bottom', etc.
category_dict = dict(zip(category_edges['source'], category_edges['target']))

print("Mapping categories to transactions...")
# Apply the mapping. If an item isn't a top or bottom, it becomes NaN
transactions['category'] = transactions['article_id'].map(category_dict)

# Drop transactions for accessories, shoes, or unrecognized items
trans_filtered = transactions.dropna(subset=['category']).copy()

# Separate Tops and Bottoms
trans_tops = trans_filtered[trans_filtered['category'].str.contains('top')].copy()
trans_bottoms = trans_filtered[trans_filtered['category'].str.contains('bottom')].copy()

# Extract just the gender word (e.g., split 'womens top' to get 'womens')
trans_tops['gender'] = trans_tops['category'].str.split(' ').str[0]
trans_bottoms['gender'] = trans_bottoms['category'].str.split(' ').str[0]

print("Finding co-occurrences (Same customer, same date)...")
# Merge to find items bought together
merged = pd.merge(
    trans_tops[['customer_id', 't_dat', 'article_id', 'gender']],
    trans_bottoms[['customer_id', 't_dat', 'article_id', 'gender']],
    on=['customer_id', 't_dat'],
    suffixes=('_top', '_bottom')
)

# Enforce Gender Matching
# CRITICAL: Only keep pairs where both items are for the same gender
valid_pairs = merged[merged['gender_top'] == merged['gender_bottom']].copy()

print("Counting frequency of matching outfits...")
# Count the frequencies 
pair_counts = valid_pairs.groupby(
    ['article_id_top', 'article_id_bottom']
).size().reset_index(name='count')

# Sort to find the most frequently bought together pairs
pair_counts = pair_counts.sort_values(['article_id_top', 'count'], ascending=[True, False])

# --- 6. Avoid the K-Core Trap! ---
# Keep the top 5 bottoms for each top, rather than just the single best 1
best_matches = pair_counts.groupby('article_id_top').head(5)

# --- 7. Format for the Knowledge Graph ---
match_edges = best_matches[['article_id_top', 'article_id_bottom', 'count']].copy()
match_edges.columns = ['source', 'target', 'weight']
match_edges['relation'] = 'best_matches_with'

print(f"Generated {len(match_edges)} highly accurate 'best_matches_with' edges.")
display(match_edges.head(10))

Mapping categories to transactions...
Finding co-occurrences (Same customer, same date)...
Counting frequency of matching outfits...
Generated 132572 highly accurate 'best_matches_with' edges.


,source,target,weight,relation
1447,0108775015,0706016002,97,best_matches_with
1446,0108775015,0706016001,91,best_matches_with
376,0108775015,0562245001,74,best_matches_with
9,0108775015,0179123001,71,best_matches_with
1289,0108775015,0688432001,70,best_matches_with
3120,0108775044,0695545001,75,best_matches_with
3199,0108775044,0706016002,72,best_matches_with
3124,0108775044,0695545007,69,best_matches_with
3198,0108775044,0706016001,55,best_matches_with
2312,0108775044,0562245050,44,best_matches_with
